<a href="https://colab.research.google.com/github/lilydeller/Math344-Project/blob/main/notebooks/04_what_it_learned.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 1

Clone repository and import functions.

In [2]:
import sys

USER = "lilydeller"
REPO = "Math344-Project"

!rm -rf {REPO}
!git clone -q https://github.com/{USER}/{REPO}.git

sys.path.insert(0, f"{REPO}/src")

from evaluate import to_matrix
from models import baseline_matrix, rank_k

Download and import the movie dataset with movie titles.

In [3]:
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

ML_DATASET = Path("ml-100k")

if not ML_DATASET.exists():
    url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
    response = requests.get(url)

    ZipFile(BytesIO(response.content)).extractall()

movie_ratings = pd.read_csv(
    ML_DATASET / "u.data", sep="\t", names=["user", "item", "rating", "time"]
)

titles = pd.read_csv(
    ML_DATASET / "u.item", sep="|", encoding="latin-1", header=None, usecols=[1]
)[1].to_numpy()

print(len(titles), titles[49])

1682 Star Wars (1977)


# Part 2

Create ratings matrix from full dataset and using it to calculate if a movie is popular it has at least 50 ratings.

In [4]:
R = to_matrix(movie_ratings)
known = ~np.isnan(R)
B = baseline_matrix(R)
X = np.where(known, R - B, 0.0)
U, s, Vt = np.linalg.svd(X, full_matrices=False)

n_ratings = known.sum(axis=0)
popular = n_ratings >= 50
print("popular movies: ", popular.sum())

popular movies:  603


# Part 3

Sorts a row of movies' scores, calculated based on a hidden factor, in ascending order. Then, it prints the first/top and last/bottom n scores alongside each movie title, representing the two contrasting ends identified by the factor.

In [5]:
def show_factor(f, n=6):
#Popular movies at the two ends of hidden factor f (f = 1, 2, ...)
  v = Vt[f - 1]
  idx = np.where(popular)[0]
  order = idx[np.argsort(v[idx])]
  print(f"Factor {f}, one end:")
  for m in order[:n]:
    print(f" {v[m]:+.3f} {titles[m]}")
  print(f"Factor {f}, other end:")
  for m in order[::-1][:n]:
    print(f" {v[m]:+.3f} {titles[m]}")

show_factor(2, 8)

Factor 2, one end:
 -0.099 Independence Day (ID4) (1996)
 -0.092 Twister (1996)
 -0.077 Eraser (1996)
 -0.074 Air Force One (1997)
 -0.067 Saint, The (1997)
 -0.066 Phenomenon (1996)
 -0.062 Ghost and the Darkness, The (1996)
 -0.061 Murder at 1600 (1997)
Factor 2, other end:
 +0.203 Star Wars (1977)
 +0.199 Pulp Fiction (1994)
 +0.173 Godfather, The (1972)
 +0.168 Empire Strikes Back, The (1980)
 +0.151 Fargo (1996)
 +0.146 Return of the Jedi (1983)
 +0.134 2001: A Space Odyssey (1968)
 +0.133 Jaws (1975)


# Part 4

The code finds any movies with the given word somewhere in the title. Enumerate gives each title with its position, with .lower() making it ignore casing. m + 1 is printed instead of m because while m starts at 0, the movie ID starts at 1.

In [10]:
def find(word):
  """Print the ID and title of every movie whose title contains word."""
  for m, t in enumerate(titles):
    if word.lower() in t.lower():
      print(m + 1, t)

find("alien")

176 Aliens (1986)
183 Alien (1979)
343 Alien: Resurrection (1997)
665 Alien 3 (1992)


Below is a user representing "me"

In [27]:
my_ratings = {50: 5, 172: 4, 181: 5, 174: 4, 210: 3, 173: 5, 27: 2,
              151: 4, 29: 2, 114: 1, 250: 1, 204: 4, 82: 5, 132: 3,
              257: 2}

The code below adds the my_ratings as a new user (me) and then retrains the database. The predictions for "me" are contained in P[-1], the last row.

In [28]:
me = np.full((1, R.shape[1]), np.nan)
for movie, stars in my_ratings.items():
  me[0, movie - 1] = stars
R_me = np.vstack([R, me])
print(R_me.shape)

known_me = ~np.isnan(R_me)
B_me = baseline_matrix(R_me)
X_me = np.where(known_me, R_me - B_me, 0.0)
U, s, Vt = np.linalg.svd(X_me, full_matrices=False)
P = rank_k(U, s, Vt, 10) + B_me

(944, 1682)


can_pick is true for popular movies that have not been rated by me, np.where keeps the predicted scores of movies that can be picked and sets the score of movies that can't to -infinity, the two "set"s keeps movies that are in both lists. The code compares the top 10 results from Model 2 with the Baseline. It selects the 10 movies with the highest predicted ratings from each and counts how many appear in both.

In [29]:
mine = P[-1]
base = B_me[-1]
can_pick = popular & np.isnan(R_me[-1])

def top(scores, n=10):
  """The n highest scoring movies among those we may recommend."""
  return np.argsort(-np.where(can_pick, scores, -np.inf))[:n]

print("Model 2 recommends:")
for m in top(mine):
  print(f" {mine[m]:.2f} {titles[m]}")
print("Baseline recommends:")
for m in top(base):
  print(f" {base[m]:.2f} {titles[m]}")
print("in both lists:", len(set(top(mine)) & set(top(base))))

Model 2 recommends:
 4.28 Close Shave, A (1995)
 4.23 Schindler's List (1993)
 4.22 Shawshank Redemption, The (1994)
 4.20 Wrong Trousers, The (1993)
 4.17 Casablanca (1942)
 4.15 Usual Suspects, The (1995)
 4.08 Good Will Hunting (1997)
 4.06 12 Angry Men (1957)
 4.06 Titanic (1997)
 4.04 Godfather, The (1972)
Baseline recommends:
 4.27 Close Shave, A (1995)
 4.19 Wrong Trousers, The (1993)
 4.16 Schindler's List (1993)
 4.16 Shawshank Redemption, The (1994)
 4.13 Casablanca (1942)
 4.11 Usual Suspects, The (1995)
 4.10 Good Will Hunting (1997)
 4.04 Titanic (1997)
 4.04 12 Angry Men (1957)
 4.03 Rear Window (1954)
in both lists: 9


The below code prints out a list of what movies were moved up most and which were moved down.

In [30]:
personal = mine - base
print("Moved up the most for me:")
for m in top(personal, 6):
  print(f" {personal[m]:+.2f} {titles[m]}")
print("Moved down most for me:")
for m in top(-personal, 6):
  print(f" {personal[m]:+.2f} {titles[m]}")

Moved up the most for me:
 +0.10 It's a Wonderful Life (1946)
 +0.10 Braveheart (1995)
 +0.10 Terminator, The (1984)
 +0.10 Star Trek: The Wrath of Khan (1982)
 +0.09 Jaws (1975)
 +0.09 Star Trek: First Contact (1996)
Moved down most for me:
 -0.22 Pulp Fiction (1994)
 -0.10 Rock, The (1996)
 -0.09 Trainspotting (1996)
 -0.08 Brazil (1985)
 -0.08 Twister (1996)
 -0.08 Annie Hall (1977)


# Part 5

Without being told any genres, the SVD discovered a tendency towards sci-fi and adventure movies but there is no clear shared genre in what was pushed down; from 15 of my ratings, it moved movies such as Star Trek and Terminator up with movies like Pulp Fiction and Trainspotting moved down the most, and 9 of the 10 top picks were the same crowd favorites the simple average recommends.